A notebook to clean the test data and calculate z scores for each test parameter.

In [ ]:
import pandas as pd
import numpy as np
import math

df = pd.read_csv("raw/4Q24-1Q26.csv")#, header=0) 
df.head()
#df.value_counts()

showing that there are duplicates when case-insensitive; we will ask Mike how he wants to approach this.

In [ ]:

print(len(df['SerialNumber'].str.upper().unique()))
print(len(df['SerialNumber'].unique()))

Cleaning the serial numbers by making all uppercase:

In [ ]:
df['SerialNumber'] = df['SerialNumber'].str.upper()

In [ ]:
# Clean column, turn "-" into NaN
def to_num(s):
    s = s.astype(str).str.strip()
    s = s.replace({"-": np.nan, "": np.nan})
    s = s.str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")




This method uses the range rule of thumb (that the mean is the average of the min and max and the standard deviation is the range divided by 4). This assumes normality and may not be accurate.

In [ ]:
# currently this function is not being used. instead we are using zscore_empirical(), defined in a below cell.
def zscore_range_rule_of_thumb(df):

    # Convert columns to num
    val = to_num(df["Value"])
    mn  = to_num(df["Minimum"])
    mx  = to_num(df["Maximum"])

    # mean and std (based on min/max)
    df["Row_Mean"] = (mn + mx) / 2
    df["Row_StdDev"] = (mx - mn) / math.sqrt(12)

    # Compute z-score only for rows where Value is numeric and std is nonzero
    mask = val.notna() & df["Row_StdDev"].notna() & df["Row_StdDev"].ne(0)

    df["Value_z"] = np.where(
        mask,
        (val - df["Row_Mean"]) / df["Row_StdDev"],
        np.nan
    )
    return df

#df.head(20)

Drop any columns where  Value is NA. Also clean the commas in values, max, and min.

Also dropped where Maximum == Minimum, as these results are for record only/can be ignored.

In [ ]:
df.dropna()
df = df[df['Maximum'] != df['Minimum']] # drop values that are for record only.

df['Value'] = to_num(df['Value'])
df['Maximum'] = to_num(df['Maximum'])
df['Minimum'] = to_num(df['Minimum'])


This function calculates the z-score of each Parameter using the values of the sample data.

In [ ]:
def zscore_empirical(df):
    # get unique parameter labels
    parameter_labels = df['Parameter'].unique()

    # for each, calculate the standard deviation and mean. 

    for parameter in parameter_labels:
        matches = to_num(df[df['Parameter'] == parameter]['Value']).astype('float')

        mean = matches.mean()
        std = matches.std()

        # z score = (value - mean)/std
        if std != 0:
            df.loc[df['Parameter']==parameter, ['z_score']] = (df['Value'] - mean)/std
        else:
            # can't compute a z score, but count test as a "pass" (0 z score)
            df.loc[df['Parameter']==parameter, ['z_score']] = 0.0

    return df

df = zscore_empirical(df)



In [ ]:
df.head(5)

Add a column that says whether the parameter is a pass or a fail. The criteria for a fail is a z score of absolute value >= 2 or a value that is outside of the minimum or maximum.

In [ ]:
df['Passed'] = ((abs(df['z_score']) < 2) & (df['Value'] <= df['Maximum']) & (df['Value'] >= df['Minimum']))
((abs(df['z_score']) < 2) & (df['Value'] <= df['Maximum']) & (df['Value'] >= df['Minimum'])).value_counts()
#df.head()

Showing how many tests are classified as passes / fails for if we used 1 standard deviations away as a fail instead of 2 away.

In [ ]:
((abs(df['z_score']) < 1) & (df['Value'] <= df['Maximum']) & (df['Value'] >= df['Minimum'])).value_counts()

### Pivot so that we are seeing each SerialNumber's passed and failed tests.
This first one groups by PARAMETER group, and holds the value of Passed. So for each unique transmission number, it will hold a boolean showing if each specific test value passed or didn't pass

In [ ]:
# group by the PARAMETER group, and hold the value of Passed
parameter_pivot_table = df.pivot_table(
    index='SerialNumber',
    columns='Parameter',
    values='Passed',
    aggfunc='any'
)



In [ ]:
# see the dimensions
print('rows:', parameter_pivot_table.shape[0], 'cols:', parameter_pivot_table.shape[1])
# how many NaNs are there for each test category?
parameter_pivot_table.isna().sum().sort_values().tail()

We can also group by the TEST group (which has multiple PARAMETERS under each one), and hold the percentage of passed tests. This gets us down from 422 columns to 108, helping with our dimensionality reduction issue.


In [ ]:
test_pivot_table = df.pivot_table(
    index='SerialNumber',
    columns='Test',
    values='Passed',
    aggfunc='mean'
)

test_pivot_table

As per Mike's instructions, remove Test categories with >90% missing values.

In [ ]:
threshold = test_pivot_table.shape[0] * 0.1
test_df_dropped = test_pivot_table.dropna(axis=1, thresh=threshold)


In [ ]:
# see the dimensions
print('Before: rows:', test_pivot_table.shape[0], 'cols:', test_pivot_table.shape[1])
print('After: rows:', test_df_dropped.shape[0], 'cols:', test_df_dropped.shape[1])

# how many NaNs are there for each test category?
test_df_dropped.isna().sum().sort_values()

In [ ]:
df.to_csv("clean/test_data_zscore.csv", index=False, float_format="%.6f")
#test_pivot_table.to_csv("clean/pivoted_test_data.csv", index=False, float_format="%.6f")
#parameter_pivot_table.to_csv("clean/pivoted_parameter_data.csv", index=False, float_format="%.6f")
test_df_dropped.to_csv("clean/test_data.csv", index=False, float_format="%.6f")
print("Done")